# Day 51 — Advanced feature engineering: target encoding & leakage prevention
Objectives:
- Understand target/mean encoding for high-cardinality categoricals.
- Prevent leakage with KFold schemes.
- Integrate into sklearn Pipelines.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','embarked','fare','age'])
X = df[['sex','class','embarked','fare','age']]
y = df['survived']
Xtr,Xte,ytr,yte = train_test_split(X,y, stratify=y, random_state=42)


## KFold target encoding utility (no leakage)
For each fold, compute means on train folds and apply to val fold only.

In [ ]:
def kfold_target_encode(cat: pd.Series, y: pd.Series, n_splits=5, smoothing=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    out = pd.Series(index=cat.index, dtype=float)
    global_mean = y.mean()
    for tidx, vidx in skf.split(cat, y):
        trc, trY = cat.iloc[tidx], y.iloc[tidx]
        means = trY.groupby(trc).mean()
        counts = trY.groupby(trc).size()
        smooth = (means * counts + global_mean * smoothing) / (counts + smoothing)
        out.iloc[vidx] = cat.iloc[vidx].map(smooth).fillna(global_mean)
    return out.fillna(global_mean)

Xe = Xtr.copy()
for col in ['sex','class','embarked']:
    Xe[col + '_te'] = kfold_target_encode(Xtr[col], ytr)
Xe[['sex_te','class_te','embarked_te']].head()


## Compare baseline One-Hot vs Target Encoding features
(Demonstration: combine OHE for small cats + TE for high-cardinality if present.)

In [ ]:
from sklearn.metrics import roc_auc_score
# Baseline OHE
ohe = ColumnTransformer([('ohe', OneHotEncoder(handle_unknown='ignore'), ['sex','class','embarked'])], remainder='passthrough')
pipe_ohe = Pipeline([('pre', ohe), ('clf', LogisticRegression(max_iter=1000))])
pipe_ohe.fit(Xtr,ytr); auc_ohe = roc_auc_score(yte, pipe_ohe.predict_proba(Xte)[:,1])
# TE approach
Xtr_te = Xtr.copy(); Xte_te = Xte.copy()
for c in ['sex','class','embarked']:
    Xtr_te[c+'_te'] = kfold_target_encode(Xtr[c], ytr)
    # map train means to test
    m = pd.concat([Xtr[c], ytr], axis=1).groupby(c)['survived'].mean()
    Xte_te[c+'_te'] = Xte[c].map(m).fillna(ytr.mean())
cols = ['fare','age','sex_te','class_te','embarked_te']
clf = LogisticRegression(max_iter=1000).fit(Xtr_te[cols], ytr)
auc_te = roc_auc_score(yte, clf.predict_proba(Xte_te[cols])[:,1])
auc_ohe, auc_te


## Learner exercises and progressive hints

1. Add K-fold target encoding to a scikit-learn pipeline through a custom
   transformer or `FunctionTransformer`.
2. Add an appropriate prior and smoothing; experiment with `n_splits`.
3. Compare ROC AUC with one-hot encoding across multiple seeded train/test
   splits.

### Progressive hints

1. A robust custom transformer needs distinct fitting and transform behavior.
   During training, generate out-of-fold values; for new rows, use a mapping fit
   only on training data. Write down index-alignment rules first.
2. Blend a category mean with the global prior using its support count. Test an
   unseen category and a one-row category deliberately.
3. Reuse each split for both methods and report score differences per seed, not
   only the best run.

The notebook's `kfold_target_encode` is an instructional utility, not a
drop-in production transformer. Its Series indexes and positional fold indexes
must remain aligned.

### Additional mastery practice

Implement target-derived features with row-level lineage. Training encodings must be out-of-fold; validation, test, and future rows use mappings fitted only on prior data.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Out-of-fold invariant:** Create a unique category for every training row and show that a leaky full-data target mean reproduces each label. Then prove that your out-of-fold encoder falls back to the prior instead.
   **Progressive hint:** For a category absent from the fold's training partition, there is no valid category statistic; use the fold training prior.
5. **Unknown and missing categories:** Define distinct policies for a missing category, an unseen category, and a known category with one observation. Write tests for all three.
   **Progressive hint:** Normalize missing values to an explicit sentinel if missingness is a category; unseen categories generally receive the training global prior.
6. **Temporal leakage:** Design target encoding for timestamped events where later labels cannot inform earlier rows. Compare random K-fold encoding with an expanding-time implementation.
   **Progressive hint:** Sort by event time and compute each row's category statistics from strictly earlier labeled rows; handle ties deliberately.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Out-of-fold invariant


# Practice 5 — Unknown and missing categories


# Practice 6 — Temporal leakage
